# wardowl paper, experiment X1, run 2: under-retrieval under permission filters

Run 1 (notebook `notebook11401c38ff`) showed that most of its pgvector searches never touched the HNSW index: after five executions PostgreSQL switched them to a generic plan that searches exactly. Run 2 follows amendment 2 of `PREREGISTRATION.md`: it plans every P-a to P-e search with its own values (`pg-custom`), records the plan every search executed, replays the original run's pgvector pass (`p-x0`, `p-x0v`), fixes the `one-segment` Qdrant layout, and runs the whole pre-registered design (all 1,489 questions, three passes, three builds) with the scripts of [`benchmarks/paper`](https://github.com/muzakkirhussain011/wardowl/tree/paper/x1-underretrieval/benchmarks/paper).

1. **Smoke**: a small corpus, every setting and code path, then its statistics. If it fails, the notebook stops here.
2. **pgvector**: three builds of the default index and three of m=32, ef_construction=200, the replays first.
3. **Qdrant**: five layouts, three builds each.
4. **Statistics**: intervals, paired tests, the pre-registered verdicts and P5.

**No GPU needed.** Run 2 is designed for a CPU-only session: nothing in it uses a GPU once the chunk embeddings exist. **Notebook settings:** Accelerator **None**, **Internet on**. **Input:** add the output of run 1 (`notebook11401c38ff`) or of the original run (`notebookd8377caca8`) with *Add Input*; X1 copies their `cache/chunks-*.npy` (and run 1's query vectors). Without an input, X1 embeds the 107,540 chunks on the CPU, about an hour (`ALLOW_CPU_EMBEDDING`); with a GPU T4 that takes about ten minutes.

**Time.** The whole design takes about 11 hours in one session, more than one Kaggle session allows. When the session ends, start a new version with this notebook's earlier output added as input: the X1 cells resume where it stopped. Faster: run it as parallel Kaggle sessions with `x1_run2_kaggle.py` (README, "Run 2"), as the paper's run 2 did.

Outputs, in `/kaggle/working`: `x1run2-smoke/`, `x1run2/` (with `x1run2/stats/report.md`). Each setting is written as soon as it finishes, so a crash keeps what was done.

In [ ]:
import time

SESSION_START = time.time()  # Kaggle stops a session 12 hours after it starts
BRANCH = "paper/x1-underretrieval"  # the scripts and the wardowl under test come from this branch
SRC = "/tmp/wardowl-src"  # noqa: S108 - the clone; its scripts run, the wardowl they import is installed
HOURS = 8.5  # no X1 setting starts after this many hours of the session
PGVECTOR_HOURS = 6.5  # nor a pgvector setting after this many hours of its own cell, so Qdrant gets its turn
ALLOW_CPU_EMBEDDING = True  # without an attached cache and without a GPU, embed the corpus on the CPU (about 1 h)

In [ ]:
!rm -rf {SRC} && git clone -q --branch {BRANCH} https://github.com/muzakkirhussain011/wardowl.git {SRC}
!git -C {SRC} log -1 --format="%H %cs %s"
!pip install -q "wardowl[pgvector,qdrant] @ git+https://github.com/muzakkirhussain011/wardowl@{BRANCH}" "qdrant-client>=1.16" "sentence-transformers>=3" "datasets>=2.19" psutil pyarrow scipy

In [ ]:
# PostgreSQL with pgvector 0.8.7, built from source as in the original run: Kaggle has no Docker. Qdrant 1.19.2 is
# started and stopped by X1 itself, with benchmarks/kaggle_stores.sh.
!PG_TUNE=1 bash {SRC}/benchmarks/kaggle_setup.sh

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from IPython.display import Markdown, display

CODES: dict[str, int] = {}  # each step's exit code: 0 done, 1 done with failures, 3 a leak stopped it
DEADLINE = str(SESSION_START + HOURS * 3600)
CPU = ["--allow-cpu-embedding"] if ALLOW_CPU_EMBEDDING else []


def step(name: str, script: str, *args: str) -> int:
    """Runs one script of benchmarks/paper, printing its output as it goes, and returns its exit code."""
    command = [sys.executable, f"{SRC}/benchmarks/paper/{script}", *args]
    print("$", " ".join(command), flush=True)
    pipe = {"stdout": subprocess.PIPE, "stderr": subprocess.STDOUT, "text": True}
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    with subprocess.Popen(command, env=env, **pipe) as run:  # noqa: S603
        for line in run.stdout:
            print(line, end="", flush=True)
    CODES[name] = run.returncode
    print(f"{name}: exit code {run.returncode}", flush=True)
    return run.returncode


def show(path: str) -> None:
    page = Path(path)
    if page.exists():
        display(Markdown(page.read_text(encoding="utf-8")))
    else:
        print(f"{path} was not written: see the step's output above")

In [ ]:
import shutil

# A later session can continue an unfinished run 2: add the earlier version of this notebook as input too. Its
# x1run2/ is copied here, and the X1 cells (--resume) skip every build it finished; a build left half done is redone.
earlier = sorted(Path("/kaggle/input").glob("**/x1run2/manifest.json"))
if earlier and not Path("/kaggle/working/x1run2").exists():
    shutil.copytree(earlier[0].parent, "/kaggle/working/x1run2", ignore=shutil.ignore_patterns("stats"))
    print(f"continuing run 2 in {earlier[0].parent}" + (f" (of {len(earlier)} found)" if len(earlier) > 1 else ""))
else:
    print("no earlier run 2 among the inputs: run 2 starts afresh")

## Smoke: every setting and code path on a small corpus

About 3,000 chunks, a dozen people, one build of each variant, two timed passes, the replays and the executed plans. A failure here stops the notebook before the long run.

In [ ]:
smoke = step("x1 smoke", "x1_underretrieval.py", "--smoke", "--resume", "--deadline", DEADLINE)
smoke_stats = step("x1 smoke statistics", "x1_stats.py", "--in", "/kaggle/working/x1run2-smoke", "--resamples", "500")
show("/kaggle/working/x1run2-smoke/stats/report.md")
if smoke or smoke_stats:
    raise RuntimeError(f"the smoke run did not pass ({CODES}): see /kaggle/working/x1run2-smoke/manifest.json")

## pgvector (section 6.2, amendment 2)

PostgreSQL + pgvector loaded as in the original run; first the two replays of the original pass, then the grid P-a to P-h on three builds of the default index and the core grid on three builds of m=32, ef_construction=200. The plan of every first-pass search is recorded; EXPLAIN (ANALYZE, BUFFERS) on every short result and on 50 fixed questions.

In [ ]:
pgvector = ["--stores", "pgvector", "--resume", "--deadline", DEADLINE, "--max-hours", str(PGVECTOR_HOURS), *CPU]
step("x1 pgvector", "x1_underretrieval.py", *pgvector)

## Qdrant (section 6.3, amendment 2)

Five layouts (as run, `is_tenant`, `is_tenant` with `m=0` and `payload_m=16`, a separate knowledge-base collection, one segment), three builds each; ACORN off, at 0.4 and at 1.0, `hnsw_ef` 64, 128 and 512, and exact search. Qdrant's search-path counters are read per person, and the points of every segment after each build.

In [ ]:
step("x1 qdrant", "x1_underretrieval.py", "--stores", "qdrant", "--resume", "--deadline", DEADLINE, *CPU)

## Statistics and the verdicts (sections 6.5 and 6.6, amendment 2)

In [ ]:
step("x1 statistics", "x1_stats.py", "--in", "/kaggle/working/x1run2")
show("/kaggle/working/x1run2/stats/report.md")
unclean = {name: code for name, code in CODES.items() if code}
print("every step finished cleanly" if not unclean else f"steps that did not finish cleanly: {unclean}")
print("see the manifests in /kaggle/working/x1run2-smoke and x1run2 for unsupported, skipped and failed settings")